In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [ ]:
# 2. Load and Preprocess the Dataset
      # Step 1 - Download the stock market dataset.

from google.colab import files
uploaded = files.upload()

import pandas as pd
stock_market = pd.read_csv("stock_market_dataset.csv")
print(stock_market.head())

Saving stock_market_dataset.csv to stock_market_dataset (4).csv
           unix        date    symbol    open    high     low   close  \
0  1.640560e+12  12-27-2021  XRP-USDT  0.9200  0.9237  0.9200  0.9226   
1  1.640480e+12  12-26-2021  XRP-USDT  0.9252  0.9334  0.9052  0.9200   
2  1.640390e+12  12-25-2021  XRP-USDT  0.9114  0.9350  0.8981  0.9252   
3  1.640300e+12  12-24-2021  XRP-USDT  0.9941  0.9966  0.8964  0.9115   
4  1.640220e+12  12-23-2021  XRP-USDT  0.9538  1.0167  0.9372  0.9941   

    Volume XRP   Volume USDT  
0    2384512.0  2.198450e+06  
1  163438501.0  1.499400e+08  
2  250074945.0  2.302303e+08  
3  567234092.0  5.377035e+08  
4  479436230.0  4.729372e+08  


In [ ]:
shape = stock_market.shape
print(shape)

(1334, 9)


In [ ]:
print("📊 Statistiques :")
print(stock_market.describe())

📊 Statistiques :
               unix         open         high          low        close  \
count  1.334000e+03  1334.000000  1334.000000  1334.000000  1334.000000   
mean   1.582978e+12     0.473318     0.495597     0.450075     0.473632   
std    3.328447e+10     0.319903     0.341216     0.297513     0.320127   
min    1.525390e+12     0.135360     0.149380     0.101290     0.135490   
25%    1.554182e+12     0.256295     0.263605     0.250050     0.256343   
50%    1.582975e+12     0.326050     0.337595     0.315500     0.326050   
75%    1.611770e+12     0.565155     0.599938     0.528917     0.566190   
max    1.640560e+12     1.833960     1.966890     1.652430     1.834680   

         Volume XRP   Volume USDT  
count  1.334000e+03  1.334000e+03  
mean   3.567016e+08  2.356279e+08  
std    5.914574e+08  4.840987e+08  
min    2.384512e+06  2.198450e+06  
25%    6.448872e+07  1.894323e+07  
50%    1.480822e+08  4.136179e+07  
75%    3.779833e+08  2.519194e+08  
max    8.608358e+09

In [ ]:
info = stock_market.info()
print(info)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1334 entries, 0 to 1333
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   unix         1334 non-null   float64
 1   date         1334 non-null   object 
 2   symbol       1334 non-null   object 
 3   open         1334 non-null   float64
 4   high         1334 non-null   float64
 5   low          1334 non-null   float64
 6   close        1334 non-null   float64
 7   Volume XRP   1334 non-null   float64
 8   Volume USDT  1334 non-null   float64
dtypes: float64(7), object(2)
memory usage: 93.9+ KB
None


In [ ]:
print(stock_market.tail())

              unix      date    symbol     open     high      low    close  \
1329  1.525740e+12  5-8-2018  XRP-USDT  0.82490  0.84802  0.79200  0.80667   
1330  1.525650e+12  5-7-2018  XRP-USDT  0.86482  0.86886  0.80000  0.82490   
1331  1.525560e+12  5-6-2018  XRP-USDT  0.90280  0.91800  0.83774  0.86483   
1332  1.525480e+12  5-5-2018  XRP-USDT  0.88980  0.93500  0.88800  0.90280   
1333  1.525390e+12  5-4-2018  XRP-USDT  0.50000  1.50000  0.50000  0.88990   

       Volume XRP  Volume USDT  
1329  12971303.18  10571844.13  
1330  17303486.40  14192279.73  
1331  16002035.80  13997141.56  
1332  16816165.30  15282000.08  
1333  20890213.82  18946724.69  


In [ ]:
# Drop unnecessary columns ...
stock_market = stock_market.drop(columns=['unix', 'symbol'])

# ... and create a "target" column for the next day’s closing price => création de la colonne "target" = close du jour suivant
stock_market['target'] = stock_market['close'].shift(-1)

# Supprimer la dernière ligne, car target = NaN à la fin
stock_market = stock_market.dropna()


In [ ]:
# Traitement de la colonne "date" :
    # Convertir date en datetime Python
stock_market['date'] = pd.to_datetime(stock_market['date'])

# Création de nouvelles colonnes à partir de 'date'
stock_market['year'] = stock_market['date'].dt.year
stock_market['month'] = stock_market['date'].dt.month
stock_market['day'] = stock_market['date'].dt.day
stock_market['day_of_week'] = stock_market['date'].dt.dayofweek  # 0=lundi, 6=dimanche
# Ces informations peuvent capturer des effets saisonniers ou hebdomadaires (ex. : chute des cryptos le week-end).

In [ ]:
# Supprimer la colonne 'date' (encore au format datetime)
stock_market = stock_market.drop(columns=['date'])

In [ ]:
info = stock_market.info()
print(info)

<class 'pandas.core.frame.DataFrame'>
Index: 1333 entries, 0 to 1332
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   open         1333 non-null   float64
 1   high         1333 non-null   float64
 2   low          1333 non-null   float64
 3   close        1333 non-null   float64
 4   Volume XRP   1333 non-null   float64
 5   Volume USDT  1333 non-null   float64
 6   target       1333 non-null   float64
 7   year         1333 non-null   int32  
 8   month        1333 non-null   int32  
 9   day          1333 non-null   int32  
 10  day_of_week  1333 non-null   int32  
dtypes: float64(7), int32(4)
memory usage: 104.1 KB
None


In [ ]:
# Normalize the dataset using MinMaxScaler.
    # 1. Importer le scaler
from sklearn.preprocessing import MinMaxScaler
    # 2. Initialiser le scaler
scaler = MinMaxScaler()
    # 3. Appliquer la normalisation pour toutes les colonnes du DataFrame
stock_market_scaled = scaler.fit_transform(stock_market)



In [ ]:
# 3. Prepare the Dataset for Training
      # Split the dataset into training, validation, and testing sets.
      # Create a custom PyTorch Dataset class to handle the data.
      # Use DataLoader to create iterable datasets for training and evaluation.

import numpy as np

# Diviser les données temporelles (ex : 70% train, 15% val, 15% test)
n = len(stock_market_scaled)
train_size = int(n * 0.7)
val_size   = int(n * 0.15)

train_data = stock_market_scaled[:int(0.7*len(stock_market_scaled))]
val_data   = stock_market_scaled[int(0.7*len(stock_market_scaled)):int(0.85*len(stock_market_scaled))]
test_data  = stock_market_scaled[int(0.85*len(stock_market_scaled)):]


In [78]:
# Créer une classe PyTorch Dataset personnalisée
# Cette classe prépare les séquences temporelles pour un LSTM :
# Chaque entrée est une séquence de window_size jours, et le label est la valeur du jour suivant.

import torch
from torch.utils.data import Dataset

class StockDataset(Dataset):
    def __init__(self, data, window_size=20):
        self.window_size = window_size
        self.data = torch.tensor(data, dtype=torch.float32)

    def __len__(self):
        return len(self.data) - self.window_size

    def __getitem__(self, idx):
        x = self.data[idx:idx + self.window_size, :-1]  # toutes les features sauf target
        y = self.data[idx + self.window_size, -1]       # target du jour suivant
        return x, y

In [79]:
# Créer les DataLoader (on utilise les datasets personnalisés avec DataLoader, pour l’entraînement).
from torch.utils.data import DataLoader

window_size = 20
batch_size = 32

train_dataset = StockDataset(train_data, window_size)
val_dataset   = StockDataset(val_data, window_size)
test_dataset  = StockDataset(test_data, window_size)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size)
test_loader  = DataLoader(test_dataset, batch_size=batch_size)#

In [ ]:

# Dataset
class StockDataset(Dataset):
    def __init__(self, data, window_size=20):
        self.window_size = window_size
        self.data = torch.tensor(data.values, dtype=torch.float32)

    def __len__(self):
        return len(self.data) - self.window_size

    def __getitem__(self, idx):
        x = self.data[idx:idx + self.window_size, :-1]
        y = self.data[idx + self.window_size, -1]
        return x, y

# DataLoader
train_loader = DataLoader(StockDataset(train_data), batch_size=32, shuffle=True)
val_loader   = DataLoader(StockDataset(val_data), batch_size=32)
test_loader  = DataLoader(StockDataset(test_data), batch_size=32)

In [81]:
# 4. Define the LSTM Model
      # Create an LSTM model using PyTorch.
      # Define the model architecture, including GRU layers, dropout, and a dense layer.

# Objectif - créer un modèle PyTorch de type GRU (ou LSTM) :
      # avec 1 ou plusieurs couches GRU
      # du Dropout pour éviter l'overfitting
      # une Linear pour la prédiction finale

  # Step 1 : Importation des modules nécessaires
import torch
import torch.nn as nn

In [82]:
# 4. Define the LSTM Model
  # Step 2 : Définir le modèle GRU
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super(GRUModel, self).__init__()

        self.gru = nn.GRU(input_size=input_size,
                          hidden_size=hidden_size,
                          num_layers=num_layers,
                          batch_first=True,
                          dropout=dropout)

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)  # 1 seule sortie (valeur prédite)

    def forward(self, x):
        # x shape: [batch_size, sequence_length, input_size]
        gru_out, _ = self.gru(x)  # on ignore le dernier état caché
        out = self.dropout(gru_out[:, -1, :])  # on prend la dernière sortie temporelle
        return self.fc(out)  # shape: [batch_size, 1]

# Hypothèses d’instanciation :
    # dataset de 10 colonnes d’entrée (open, high, low, etc. + target)
    # 2 couches GRU
    # hidden_size = 64
    # dropout = 0.3

input_size = train_data.shape[1] - 1  # toutes les colonnes sauf target
hidden_size = 64
num_layers = 2
dropout = 0.3

model = GRUModel(input_size, hidden_size, num_layers, dropout)

# Envoi du modèle sur le bon device (GPU/CPU) :
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)


GRUModel(
  (gru): GRU(10, 64, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)

In [84]:
# 5. Train the Model
  # Step 1. Set up the optimizer and loss function.
  # Step 2. Implement training and validation loops.
  # Step 3. Train the model for a specified number of epochs.

# Prerequisite
  # Configuration
import torch
import torch.nn as nn
import torch.optim as optim
  # Parameters
num_epochs = 10
learning_rate = 0.001
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [85]:
  # Step 1. Set up the model, optimizer and loss function.
model = GRUModel(input_size=train_data.shape[1]-1, hidden_size=64, num_layers=2, dropout=0.3).to(device)
criterion = nn.MSELoss()  # perte pour un problème de régression
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [86]:
  # Step 2. Implement training and validation loops.
for epoch in range(num_epochs):
    model.train()
    train_losses = []

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    # Validation
    model.eval()
    val_losses = []

    with torch.no_grad():
        for X_val, y_val in val_loader:
            X_val, y_val = X_val.to(device), y_val.to(device)
            output = model(X_val)
            val_loss = criterion(output, y_val)
            val_losses.append(val_loss.item())

    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Train Loss: {sum(train_losses)/len(train_losses):.4f}, "
          f"Val Loss: {sum(val_losses)/len(val_losses):.4f}")


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([32])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([17])) that is different to the input size (torch.Size([17, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/loss.py:610: UserWarning: Using a target size (torch.Size([20])) that is different to the input size (torch.Size([20, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input,

Epoch 1/10, Train Loss: 0.1277, Val Loss: 0.1106
Epoch 2/10, Train Loss: 0.1145, Val Loss: 0.1164
Epoch 3/10, Train Loss: 0.1156, Val Loss: 0.1127
Epoch 4/10, Train Loss: 0.1154, Val Loss: 0.1134
Epoch 5/10, Train Loss: 0.1148, Val Loss: 0.1108
Epoch 6/10, Train Loss: 0.1166, Val Loss: 0.1105
Epoch 7/10, Train Loss: 0.1155, Val Loss: 0.1106
Epoch 8/10, Train Loss: 0.1141, Val Loss: 0.1104
Epoch 9/10, Train Loss: 0.1140, Val Loss: 0.1109
Epoch 10/10, Train Loss: 0.1144, Val Loss: 0.1109


In [88]:
  # Step 3. Train the model for a specified number of epochs.

# Boucle d'entraînement
num_epochs = 20
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    model.train()  # mode entraînement
    epoch_train_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()               # 1. Réinitialise les gradients
        outputs = model(X_batch)            # 2. Prédiction du modèle
        loss = criterion(outputs, y_batch)  # 3. Calcul de la perte
        loss.backward()                     # 4. Calcul des gradients
        optimizer.step()                    # 5. Mise à jour des poids

        epoch_train_loss += loss.item()

    # Moyenne des pertes d'entraînement
    avg_train_loss = epoch_train_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # Validation
    model.eval()
    epoch_val_loss = 0.0
    with torch.no_grad():  # désactive le calcul de gradients
        for X_val, y_val in val_loader:
            X_val, y_val = X_val.to(device), y_val.to(device)
            val_outputs = model(X_val)
            val_loss = criterion(val_outputs, y_val)
            epoch_val_loss += val_loss.item()

    avg_val_loss = epoch_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)

    # Afficher les résultats
    print(f"Epoch [{epoch+1}/{num_epochs}] — "
          f"Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

Epoch [1/20] — Train Loss: 0.1131, Val Loss: 0.1129
Epoch [2/20] — Train Loss: 0.1141, Val Loss: 0.1122
Epoch [3/20] — Train Loss: 0.1138, Val Loss: 0.1105
Epoch [4/20] — Train Loss: 0.1128, Val Loss: 0.1113
Epoch [5/20] — Train Loss: 0.1134, Val Loss: 0.1146
Epoch [6/20] — Train Loss: 0.1127, Val Loss: 0.1104
Epoch [7/20] — Train Loss: 0.1135, Val Loss: 0.1125
Epoch [8/20] — Train Loss: 0.1124, Val Loss: 0.1110
Epoch [9/20] — Train Loss: 0.1126, Val Loss: 0.1105
Epoch [10/20] — Train Loss: 0.1141, Val Loss: 0.1104
Epoch [11/20] — Train Loss: 0.1131, Val Loss: 0.1104
Epoch [12/20] — Train Loss: 0.1125, Val Loss: 0.1104
Epoch [13/20] — Train Loss: 0.1131, Val Loss: 0.1107
Epoch [14/20] — Train Loss: 0.1129, Val Loss: 0.1115
Epoch [15/20] — Train Loss: 0.1131, Val Loss: 0.1111
Epoch [16/20] — Train Loss: 0.1128, Val Loss: 0.1106
Epoch [17/20] — Train Loss: 0.1133, Val Loss: 0.1105
Epoch [18/20] — Train Loss: 0.1131, Val Loss: 0.1105
Epoch [19/20] — Train Loss: 0.1140, Val Loss: 0.1105
Ep

In [89]:
# 6. Evaluate the Model
    # Calculate the R² score to evaluate the model’s performance on the test set.
    # Save the scaler object (MinMaxScaler for Scikit-learn) for enabling further predictions in the future.

    # Step 1. Evaluation – Score R² (le score R² = coefficient de détermination, mesure à quel point les prédictions du modèle sont proches des vraies valeurs).

from sklearn.metrics import r2_score

model.eval()  # passage en mode évaluation
predictions = []
true_values = []

with torch.no_grad():
    for X_test, y_test in test_loader:
        X_test = X_test.to(device)
        outputs = model(X_test).cpu().numpy()
        y_test = y_test.numpy()

        predictions.extend(outputs.flatten())
        true_values.extend(y_test.flatten())

    # R² calculation :
r2 = r2_score(true_values, predictions)
print(f"R² Score on test set: {r2:.4f}")

    # Step 2. Sauvegarder le scaler (MinMaxScaler)
        # Si tu veux plus tard faire des prédictions sur de nouvelles données, il est obligatoire d’utiliser le même scaler.
        # Pour le sauvegarder :
import joblib
joblib.dump(scaler, 'minmax_scaler.pkl')
print("Scaler saved successfully as 'minmax_scaler.pkl'")

scaler = joblib.load('minmax_scaler.pkl')

R² Score on test set: -0.0002
Scaler saved successfully as 'minmax_scaler.pkl'
